In [ ]:
library(Seurat)
library(ggplot2)
library(pheatmap)
library(dplyr)
library(RColorBrewer)
getwd()
dir.create("figures_10xHuman_8CLC")
dir.create("data_10xHuman_8CLC")

colorDict = c("8CLC"="#88535A",
              "ESC"="#F2CC8F")

In [ ]:
dataset_id <- "10xHuman_8CLC"
sample <- "DUX4"

path <- paste0("/mnt/volume_1p5T/results/STARsolo_EM_TE/",dataset_id,"/",sample,"/TE_Solo.out/Gene/")

# get filtered barcodes
STAR_path <- paste0("/mnt/TEresults/snakemake_results/results/STARoutdir/",dataset_id,"/",sample,"/best_Solo.out/Gene")
filteredBarcodes <- read.table(paste0(STAR_path, "/filtered/barcodes.tsv"))$V1 # read barcodes seleced by STARsolo

STARsolo_TE_EM_mat <- Seurat::ReadMtx(
        mtx = paste0(path, "/raw/UniqueAndMult-EM.mtx"),
        cells = paste0(path, "/raw/barcodes.tsv"),
        features = paste0(path, "/raw/features.tsv")
    ) # read matrix
STARsolo_TE_EM_mat <- STARsolo_TE_EM_mat[, filteredBarcodes] # keep only barcodes passing thresholds

nCells <- length(filteredBarcodes)
thrMinCells <- round(nCells * 0.02)
thrMinCells 

objTE_STARsolo <- Seurat::CreateSeuratObject(STARsolo_TE_EM_mat,
        project = "STARsolo_TE_EM",
        min.cells = thrMinCells, min.features = 0 
    )

objTE_STARsolo

In [ ]:
annotation_stellarscope <- read.table("annotation/annotation_stellarscope_hg38.tsv") # generated with annotation_scripts/create_annotations_hg38.Rmd
head(annotation_stellarscope)

In [ ]:
# Remove some classes of TEs (already not present in SoloTE)
classesToExclude <- c("Other", "Satellite", "Unknown", "RNA")
starsoloTEs <- Features(objTE_STARsolo)

filteredStarsoloTEs <- annotation_stellarscope[annotation_stellarscope$stellarscopeID %in% starsoloTEs, ] %>%
    dplyr::filter(!class %in% classesToExclude) %>%
    pull(stellarscopeID)

print("Percentage of TEs in removed orders:")
print(length(setdiff(starsoloTEs, filteredStarsoloTEs)) / length(starsoloTEs) * 100)

objTE_STARsolo <- objTE_STARsolo[intersect(starsoloTEs, filteredStarsoloTEs), ]

print("Summary of n counts")
print(summary(objTE_STARsolo$nCount_RNA))
print("Summary of n feature")
print(summary(objTE_STARsolo$nFeature_RNA))

print("Loci present in the annotation:")
print(table(rownames(objTE_STARsolo) %in% annotation_stellarscope$stellarscopeID))


In [ ]:
feature_metadata <- annotation_stellarscope[match(Features(objTE_STARsolo), annotation_stellarscope$stellarscopeID),]

head(feature_metadata)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5)


objTE_STARsolo@meta.data$nCount_TE <- objTE_STARsolo@meta.data$nCount_RNA 
objTE_STARsolo@meta.data$nFeature_TE <- objTE_STARsolo@meta.data$nFeature_RNA 
# Visualize QC metrics as a violin plot
VlnPlot(objTE_STARsolo, features = c("nCount_TE"), ncol = 1, 
        cols = colorDict, pt.size = 0) + theme(text=element_text(size=17))
ggsave(paste0("figures_", dataset_id, "/nCountTE_violin_STARsolo.pdf"), device='pdf')

VlnPlot(objTE_STARsolo, features = c("nFeature_TE"), ncol = 1, 
        cols = colorDict, pt.size = 0) + theme(text=element_text(size=17))
ggsave(paste0("figures_", dataset_id, "/nFeatureTE_violin_STARsolo.pdf"), device='pdf')


In [ ]:

objTE_STARsolo <- JoinLayers(objTE_STARsolo)

objTE_STARsolo <- NormalizeData(objTE_STARsolo, normalization.method = "LogNormalize", scale.factor = 10000)

gc()

objTE_STARsolo <- FindVariableFeatures(objTE_STARsolo, selection.method = "vst", nfeatures = 4000)

gc()

# Identify the 10 most highly variable genes
top10 <- head(VariableFeatures(objTE_STARsolo), 10)

# plot variable features with and without labels
plot1 <- VariableFeaturePlot(objTE_STARsolo)
plot2 <- LabelPoints(plot = plot1, points = top10, repel = TRUE)
plot2

In [ ]:

gc()
all.genes <- rownames(objTE_STARsolo)
objTE_STARsolo <- ScaleData(objTE_STARsolo) # on hvgs


In [ ]:
grep("ERVL", all.genes, value = T)[1:20]


In [ ]:

objTE_STARsolo <- RunPCA(objTE_STARsolo, features = VariableFeatures(object = objTE_STARsolo))

DimPlot(objTE_STARsolo, reduction = "pca") + NoLegend()

ElbowPlot(objTE_STARsolo)


In [ ]:
objTE_STARsolo <- FindNeighbors(objTE_STARsolo, dims = 1:13, k.param = 20)
objTE_STARsolo <- FindClusters(objTE_STARsolo, resolution = 1)
objTE_STARsolo <- RunUMAP(objTE_STARsolo, dims = 1:13)
DimPlot(objTE_STARsolo, reduction = "umap")


In [ ]:
FeaturePlot(objTE_STARsolo, reduction = "umap", features = "nCount_TE", pt.size = 0.5) + 
  theme_void() +
  theme(text=element_text(size=20))

In [ ]:

saveRDS(objTE_STARsolo, paste0("data_", dataset_id, "/STARsolo_", dataset_id, "_seuratObj.RDS"))
